In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split 
from sklearn.metrics import mutual_info_score
from sklearn.feature_extraction import DictVectorizer                
from sklearn.linear_model import LogisticRegression

In [ ]:
data = 'https://raw.githubusercontent.com/alexeygrigorev/datasets/master/course_lead_scoring.csv'
!wget $data

In [ ]:
df = pd.read_csv(data)
df.head()

In [ ]:
df.columns = df.columns.str.lower().str.replace(' ', '_')
categorical_columns = list(df.dtypes[df.dtypes == 'object'].index)
for c in categorical_columns:
    df[c] = df[c].str.lower().str.replace(' ', '_')

In [ ]:
df.isnull().sum()

In [ ]:
df.info()

In [ ]:
numerical = ['number_of_courses_viewed', 'interaction_count', 'lead_score']
categorical = ['lead_source', 'industry', 'employment_status', 'location']

df_filled = df.copy()
for col in categorical:
    df_filled[col] = df_filled[col].fillna('NA')
for col in numerical:
    df_filled[col] = df_filled[col].fillna(0.0)


In [ ]:
df.isnull().sum()

In [ ]:
df.industry.value_counts()

In [ ]:
numerical = ['number_of_courses_viewed', 'interaction_count', 'lead_score']
categorical = ['lead_source', 'industry', 'employment_status', 'location']
features = numerical + categorical
converted_value = df["converted"]

In [ ]:
df.head(9)

In [ ]:
df[numerical].corrwith(df.converted)

In [ ]:
correlation_matrix = df[numerical].corr()
correlation_matrix

In [ ]:
df_train, df_temp, y_train, y_temp = train_test_split(
    df_filled.drop('converted', axis=1), 
    df_filled['converted'], 
    test_size=0.6,  
    random_state=42, 
    stratify=df_filled['converted']
)

df_val, df_test, y_val, y_test = train_test_split(
    df_temp, y_temp,
    test_size=0.5,  
    random_state=42,
    stratify=y_temp
)


In [ ]:
df_val.isnull().sum()

In [57]:
df_train_for_mi = df_train.copy()
df_train_for_mi['converted'] = y_train.values

In [58]:
mutual_info_score(df_full_train.converted, df_full_train.interaction_count)


0.06853782382273678

In [59]:
def mutual_info_converted_score(series):
    return mutual_info_score(series, df_full_train.converted)

In [60]:
score = df_full_train[categorical].apply(mutual_info_converted_score)
score.sort_values(ascending=False)
round(score,2)

lead_source          0.03
industry             0.01
employment_status    0.02
location             0.00
dtype: float64

In [61]:
dv = DictVectorizer(sparse=False)

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(df_train[features].to_dict(orient='records'))
X_val = dv.transform(df_val[features].to_dict(orient='records')) 

In [62]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

In [63]:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train,y_train)

LogisticRegression(max_iter=1000, random_state=42, solver='liblinear')

In [64]:
model.intercept_[0]

-3.028501557889021

In [65]:
model.coef_[0].round(2)

array([-0.4 , -0.06, -0.27, -0.55, -1.74, -0.88,  0.77, -0.62, -0.5 ,
       -0.43, -0.67, -0.57, -0.12,  0.81,  2.14, -0.2 , -1.11, -0.64,
       -1.7 ,  1.15, -0.52,  0.19, -0.65, -0.47, -0.68, -0.34, -0.18,
       -0.55, -0.36,  1.3 ])

In [66]:
y_pred = model.predict_proba(X_val)[:,1]
y_pred

array([0.01940481, 0.74466382, 0.85746714, 0.78726151, 0.95403985,
       0.15713384, 0.93624384, 0.89311186, 0.10095814, 0.29764903,
       0.23577102, 0.39576639, 0.95629668, 0.77526589, 0.40333749,
       0.97937418, 0.98711725, 0.54501951, 0.98127715, 0.9155059 ,
       0.56907617, 0.84178561, 0.36813838, 0.99314749, 0.78904517,
       0.18131972, 0.72174938, 0.94385248, 0.53985815, 0.87107225,
       0.92837292, 0.73710023, 0.34288907, 0.67801883, 0.49046303,
       0.39330723, 0.92785906, 0.21432873, 0.75066032, 0.91312754,
       0.98104168, 0.67565935, 0.8181092 , 0.85842439, 0.41608823,
       0.72697891, 0.78774408, 0.44291301, 0.9109719 , 0.85895293,
       0.06977029, 0.2409786 , 0.20864616, 0.17483373, 0.17887502,
       0.50640847, 0.30931619, 0.84941068, 0.69935442, 0.23847105,
       0.21704799, 0.87257966, 0.90990166, 0.53117655, 0.03355326,
       0.10271885, 0.9535561 , 0.03072105, 0.99077028, 0.6752104 ,
       0.25604501, 0.16166573, 0.04986879, 0.00822531, 0.89152

In [67]:
y_pred_val = model.predict(X_val)

accuracy = (y_pred_val == y_val).mean()
round(accuracy, 2) 

0.82

In [68]:
accuracy

0.8200455580865603

In [69]:
baseline_accuracy = accuracy

In [70]:
accuracy

0.8200455580865603

In [71]:
features
feature_importance = {}

In [72]:
feature_importance = {}

for feature in features:  
    features_without = [f for f in features if f != feature]
    
    dv_reduced = DictVectorizer(sparse=False)
    X_train_reduced = dv_reduced.fit_transform(df_train[features_without].to_dict(orient='records'))
    X_val_reduced = dv_reduced.transform(df_val[features_without].to_dict(orient='records'))
 
    model_reduced = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model_reduced.fit(X_train_reduced, y_train)
    
    accuracy_reduced = (model_reduced.predict(X_val_reduced) == y_val).mean()
    accuracy_drop = baseline_accuracy - accuracy_reduced
    feature_importance[feature] = accuracy_drop
features_to_check = ['industry', 'employment_status', 'lead_score']
drops = {f: feature_importance[f] for f in features_to_check}
least_useful = min(drops, key=drops.get)
print("Answer:", least_useful)

Answer: industry


In [73]:
best_acc = 0
best_C = None

for C in [0.01, 0.1, 1, 10, 100]:
    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)   
    y_pred = model.predict(X_val)  
    acc = (y_pred == y_val).mean()
    print(f"C={C}: accuracy = {acc:.3f}")
    if acc > best_acc:
        best_acc = acc
        best_C = C

print(f"\nBest C: {best_C}, Accuracy: {best_acc:.3f}")

C=0.01: accuracy = 0.690
C=0.1: accuracy = 0.793
C=1: accuracy = 0.820
C=10: accuracy = 0.825
C=100: accuracy = 0.827

Best C: 100, Accuracy: 0.827
